In [ ]:
from transformers import Sam3VideoModel, Sam3VideoProcessor
from accelerate import Accelerator
import torch
from PIL import Image
import os
from pathlib import Path
from typing import Dict, List
import numpy as np
import matplotlib.pyplot as plt
import math
import cv2

In [ ]:
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def tile_masks(outputs_per_frame, n_cols):
    plt.close("all")
    
    n_rows = int(math.ceil(len(outputs_per_frame) / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.7, n_rows * 4.8), squeeze=False)
    
    axes_flat = axes.ravel()
    
    for i, ax in enumerate(axes_flat):
        if i >= len(outputs_per_frame):
            ax.axis("off")
            continue
        
        ax.imshow(batch_np[i])
        ax.axis("off")
        try:
            show_mask(outputs_per_frame[i]["masks"].cpu().numpy(), ax)
        except Exception as e:
            print(i)
            print(e)
    plt.tight_layout()
    plt.show()
    
def save_binary(path, arr):
    if arr.dtype != np.bool_:
        raise TypeError
    
    packed = np.packbits(arr.ravel())
    np.savez_compressed(path, packed=packed, shape=arr.shape)

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = Sam3VideoModel.from_pretrained("facebook/sam3", device_map={"": device})
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

extensions = Image.registered_extensions().keys()
all_frames = [
    file
    for file in Path("../../data/frames").iterdir()
    if file.suffix in extensions
]

videos: Dict[int, List[Path]] = {}
for frame in all_frames:
    videos.setdefault(int(frame.stem.partition('_')[0]), []).append(frame)

for video_idx, frames in videos.items():
    frames.sort(key=lambda path: path.stem.partition("_c")[0])

In [ ]:
video_idx = 10

frames = videos[video_idx]
imgs = []
for frame_path in frames:
    img = Image.open(frame_path).convert("RGB")
    arr = np.array(img)
    imgs.append(arr)
batch_np = np.stack(imgs, axis=0)
batch_tensor = torch.tensor(batch_np, device=device)

plt.imshow(batch_np[0])

In [ ]:
text = "scissors"

inference_session = processor.init_video_session(
    video=batch_tensor,
    inference_device=device,
    processing_device=device,
    video_storage_device=device,
)

inference_session = processor.add_text_prompt(
    inference_session=inference_session,
    text=text,
)

In [ ]:
outputs_per_frame = {}
# Pass show_progress_bar=True to display a tqdm progress bar.
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session
):
    processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed_outputs

tile_masks(outputs_per_frame, 5)

In [ ]:
all_masks = np.empty((len(outputs_per_frame), *batch_np.shape[1:-1]), dtype=np.bool_)
all_boxes = np.empty((len(outputs_per_frame), 4), dtype=np.int64)

for i, output in outputs_per_frame.items():
    mask = output['masks'][0].cpu().numpy()
    all_masks[i] = mask
    
    xyxy = output['boxes'][0].cpu().numpy()
    xywh = xyxy.copy()
    xywh[:, 2] = xyxy[:, 2] - xyxy[:, 0]
    xywh[:, 3] = xyxy[:, 3] - xyxy[:, 1]
    all_boxes[i] = xywh
    
save_binary(f"../../data/masks/{video_idx}.npz", all_masks)
np.save(f"../../data/boxes/{video_idx}.npy", all_boxes)